# المحاضرة 41: دمج المصفوفات الكثيفة والمتناثرة (الأسس الرياضية والبرمجية)
### Combining Dense and Sparse Matrix Representation

> **التوثيق والشرح النظري والمعماري الشامل:**
> راجع ملف التوثيق المخصص في مجلد الشرح: [`docs/05-hybrid-search-and-retrieval/01-combining-dense-and-sparse.md`](../../../docs/05-hybrid-search-and-retrieval/01-combining-dense-and-sparse.md)

---

## الكود النموذجي المرجعي (Production Reference Code)
يتضمن هذا الدفتر التطبيق العملي الكامل لمعادلة دمج البحث الكثيف (Dense) والبحث المتناثر (Sparse) باستخدام مكتبتي `scikit-learn` و `sentence-transformers`، مع اختبار أثر تغيير معامل الموازنة $\alpha$.


In [1]:
# 1. استدعاء المكتبات وتهيئة البيئة
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

# Initialize local dense embedding model
dense_model = SentenceTransformer("all-MiniLM-L6-v2")
embedding_dim = dense_model.get_sentence_embedding_dimension()

print("[INFO] Libraries imported successfully.")
print(f"[INFO] Dense embedding model: all-MiniLM-L6-v2 (dimensions={embedding_dim})")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[INFO] Libraries imported successfully.
[INFO] Dense embedding model: all-MiniLM-L6-v2 (dimensions=384)


C:\Users\aliashraf\AppData\Local\Temp\ipykernel_29136\2204382512.py:9: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  embedding_dim = dense_model.get_sentence_embedding_dimension()


In [2]:
# 2. إعداد الوثائق التجريبية والاستعلام من المحاضرة
corpus = [
    "LangChain helps build LLM apps.",
    "Pinecone is used for vector search.",
    "Eiffel Tower is in Paris."
]

query = "build application using LLM"

print(f"[INFO] Corpus size: {len(corpus)} documents")
for idx, doc in enumerate(corpus, 1):
    print(f"   D{idx}: '{doc}'")
print(f"\n[INFO] User Query: '{query}'")

[INFO] Corpus size: 3 documents
   D1: 'LangChain helps build LLM apps.'
   D2: 'Pinecone is used for vector search.'
   D3: 'Eiffel Tower is in Paris.'

[INFO] User Query: 'build application using LLM'


In [3]:
# 3. المسار الأول: البحث المتناثر اللفظي (Sparse TF-IDF Retrieval)
# Fit TF-IDF vectorizer on corpus and transform both corpus and query
tfidf_vectorizer = TfidfVectorizer()
corpus_tfidf = tfidf_vectorizer.fit_transform(corpus)
query_tfidf = tfidf_vectorizer.transform([query])

# Compute cosine similarity on sparse TF-IDF vectors
sparse_similarities = cosine_similarity(query_tfidf, corpus_tfidf)[0]

print(f"[INFO] Sparse vocabulary size: {len(tfidf_vectorizer.get_feature_names_out())}")
print("[INFO] Sparse TF-IDF Scores for query:")
for idx, (doc, score) in enumerate(zip(corpus, sparse_similarities), 1):
    print(f"   D{idx} Score: {score:.4f} | '{doc}'")

[INFO] Sparse vocabulary size: 15
[INFO] Sparse TF-IDF Scores for query:
   D1 Score: 0.6325 | 'LangChain helps build LLM apps.'
   D2 Score: 0.0000 | 'Pinecone is used for vector search.'
   D3 Score: 0.0000 | 'Eiffel Tower is in Paris.'


In [4]:
# 4. المسار الثاني: البحث الكثيف الدلالي (Dense Semantic Retrieval)
# Encode corpus documents and query into dense continuous vectors
corpus_embeddings = dense_model.encode(corpus)
query_embedding = dense_model.encode([query])

# Compute cosine similarity on dense vectors
dense_similarities = cosine_similarity(query_embedding, corpus_embeddings)[0]

print("[INFO] Dense Semantic Scores for query:")
for idx, (doc, score) in enumerate(zip(corpus, dense_similarities), 1):
    print(f"   D{idx} Score: {score:.4f} | '{doc}'")

[INFO] Dense Semantic Scores for query:
   D1 Score: 0.6247 | 'LangChain helps build LLM apps.'
   D2 Score: 0.0487 | 'Pinecone is used for vector search.'
   D3 Score: 0.0505 | 'Eiffel Tower is in Paris.'


In [5]:
# 5. الدمج الهجين الموزون (Convex Hybrid Score Fusion)
def compute_hybrid_scores(dense_scores, sparse_scores, alpha=0.5):
    """
    Compute hybrid score using linear convex combination:
    Score_hybrid = alpha * Score_dense + (1 - alpha) * Score_sparse
    """
    return (alpha * np.array(dense_scores)) + ((1.0 - alpha) * np.array(sparse_scores))

# Calculate hybrid score with balanced 50/50 weighting (alpha = 0.5)
alpha = 0.5
hybrid_scores = compute_hybrid_scores(dense_similarities, sparse_similarities, alpha=alpha)

# Rank documents by hybrid score in descending order
ranked_indices = np.argsort(hybrid_scores)[::-1]

print(f"[INFO] Hybrid Retrieval Results (alpha={alpha}):\n")
print(f"{'Rank':<6}{'Doc':<6}{'Dense':<10}{'Sparse':<10}{'Hybrid':<10}{'Document Text'}")
print("-" * 75)
for rank, idx in enumerate(ranked_indices, 1):
    print(f"{rank:<6}D{idx+1:<5}{dense_similarities[idx]:<10.4f}{sparse_similarities[idx]:<10.4f}{hybrid_scores[idx]:<10.4f}'{corpus[idx]}'")

[INFO] Hybrid Retrieval Results (alpha=0.5):

Rank  Doc   Dense     Sparse    Hybrid    Document Text
---------------------------------------------------------------------------
1     D1    0.6247    0.6325    0.6286    'LangChain helps build LLM apps.'
2     D3    0.0505    0.0000    0.0252    'Eiffel Tower is in Paris.'
3     D2    0.0487    0.0000    0.0244    'Pinecone is used for vector search.'


In [6]:
# 6. تجربة تغيير معامل الموازنة (Effect of Varying Alpha)
alpha_values = [1.0, 0.8, 0.5, 0.2, 0.0]

print("[INFO] Evaluating Document 1 (LangChain) scores across varying alpha:")
print(f"{'Alpha':<8}{'Mode':<20}{'D1 Score':<12}{'Rank 1 Doc'}")
print("-" * 55)

for a in alpha_values:
    h_scores = compute_hybrid_scores(dense_similarities, sparse_similarities, alpha=a)
    top_doc = f"D{np.argmax(h_scores) + 1}"
    mode = "Pure Dense" if a == 1.0 else ("Pure Sparse" if a == 0.0 else f"Hybrid ({int(a*100)}/{int((1-a)*100)})")
    print(f"{a:<8.1f}{mode:<20}{h_scores[0]:<12.4f}{top_doc}")

[INFO] Evaluating Document 1 (LangChain) scores across varying alpha:
Alpha   Mode                D1 Score    Rank 1 Doc
-------------------------------------------------------
1.0     Pure Dense          0.6247      D1
0.8     Hybrid (80/19)      0.6262      D1
0.5     Hybrid (50/50)      0.6286      D1
0.2     Hybrid (20/80)      0.6309      D1
0.0     Pure Sparse         0.6325      D1


In [7]:
# 7. التحقق البرمجي النهائي (Assertions)
assert ranked_indices[0] == 0, "Document 1 must be ranked #1 in hybrid retrieval"
assert hybrid_scores[0] > hybrid_scores[1], "D1 hybrid score must exceed D2"
assert hybrid_scores[0] > hybrid_scores[2], "D1 hybrid score must exceed D3"
assert np.all((hybrid_scores >= 0.0) & (hybrid_scores <= 1.0)), "All scores must be bounded in [0, 1]"

print("[SUCCESS] All hybrid search assertions passed successfully.")

[SUCCESS] All hybrid search assertions passed successfully.


---
## مساحة التجربة والتطبيق العملي
استخدم الخلية التالية لتجربة نصوص واستعلامات مخصصة، أو اختبار تأثير تعديل معامل $\alpha$ على ترتيب الوثائق.

In [8]:
# مساحة التجربة والتطبيق العملي:
# جرب إدخال وثائق واستعلامات جديدة وحساب الدرجات الهجينة هنا:

